<a href="https://colab.research.google.com/github/dvalverdes/prueba_tecnica_DanielValverdeSalazar/blob/main/bloque4_kpi_framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bloque 4 — KPI Framework para Programa de Mejora de Productividad de Tiendas

## Objetivo del framework
Diseñar un marco de KPIs desde cero para acompañar un programa de mejora de productividad de tiendas.  
El framework busca balancear tres dimensiones clave del negocio:

1. **Productividad de tienda**
2. **Experiencia del cliente**
3. **Desempeño de proveedor**

Además, incluye:
- al menos un **leading indicator**
- al menos un **KPI compuesto**
- una **North Star Metric**
- reglas para detectar problemas de calidad en los datos

---

## North Star Metric del programa

### **North Star Metric: GMV por m² ajustado por disponibilidad**
**Definición:** mide cuánto valor comercial genera cada metro cuadrado de tienda, ajustando parcialmente por la disponibilidad operativa del surtido.

### ¿Por qué la elijo?
Porque el programa está enfocado en **productividad de tiendas**, y esta métrica:

- conecta ventas con uso eficiente del espacio
- evita mirar solo volumen bruto
- obliga a considerar que una tienda no es realmente “productiva” si vende bien pero tiene problemas frecuentes de disponibilidad
- resume de forma ejecutiva el balance entre rendimiento comercial y ejecución operativa

### Fórmula conceptual
```text
GMV por m² ajustado por disponibilidad
= (GMV / m²) * Índice de Disponibilidad

| KPI                                                   | Definición exacta                                                                                                 | Fórmula                                                                                                                                          | Frecuencia                | Fuente de datos                                           | Target sugerido                                                  | ¿Cómo detectas si el dato está mal?                                                                                               |
| ----------------------------------------------------- | ----------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------ | ------------------------- | --------------------------------------------------------- | ---------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------- |
| **1. GMV por m²**                                     | Mide cuánto GMV genera una tienda por cada metro cuadrado de superficie                                           | `GMV total del período / size_sqm`                                                                                                               | Semanal y mensual         | `transactions` + `stores`                                 | Estar por encima del percentil 50 de su formato; alerta bajo p25 | Validar que `size_sqm > 0`, que la tienda exista, que no haya GMV negativo no justificado y que no falten fechas de venta         |
| **2. Ticket promedio**                                | Valor promedio por transacción en una tienda o grupo de tiendas                                                   | `GMV / número de transacciones`                                                                                                                  | Diario, semanal y mensual | `transactions`                                            | Crecimiento YoY positivo y estable por formato                   | Revisar duplicados en `transaction_id`, nulos en `total_amount`, tickets extremos por error de carga                              |
| **3. Transacciones por m²**                           | Mide intensidad de tráfico/compras en relación con el tamaño de tienda                                            | `número de transacciones / size_sqm`                                                                                                             | Semanal y mensual         | `transactions` + `stores`                                 | Mantenerse por encima del promedio del formato                   | Validar unicidad de `transaction_id`, `size_sqm > 0`, y comparar contra histórico para detectar spikes imposibles                 |
| **4. Tasa de recompra loyalty mes 1**                 | Porcentaje de clientes loyalty de una cohorte que vuelve a comprar en el mes siguiente a su primera compra        | `clientes activos en mes 1 / tamaño de cohorte`                                                                                                  | Mensual                   | `transactions` (`loyalty_card`, `customer_id`)            | > 65% como referencia inicial; revisar por cohorte               | Verificar consistencia entre `customer_id` y `loyalty_card`, evitar clientes duplicados o IDs vacíos mal parseados                |
| **5. Disponibilidad operativa (Leading Indicator)**   | Porcentaje de tienda-item críticos sin gaps relevantes de venta que sugieran posible quiebre                      | `1 - (items críticos con gap / total items críticos monitoreados)`                                                                               | Diario y semanal          | `transactions` + `transaction_items` + `products`         | > 95%                                                            | Confirmar que los gaps no provengan de productos de baja rotación; revisar fechas faltantes, late arrivals y errores de ingestión |
| **6. GMROI por proveedor-categoría**                  | Retorno del margen bruto sobre el costo invertido para cada proveedor y categoría                                 | `Margen Bruto / Costo Total`                                                                                                                     | Semanal y mensual         | `transaction_items` + `products` + `vendors`              | > 1.2 como umbral saludable; alerta < 1                          | Validar `cost > 0`, `unit_price >= 0`, cantidades válidas y joins correctos entre producto y proveedor                            |
| **7. % GMV en promoción con uplift**                  | Porcentaje del GMV promocional que proviene de promociones con evidencia de uplift y no solo descuento            | `GMV de promos con uplift / GMV total promocional`                                                                                               | Mensual                   | `transactions` + `transaction_items` + `store_promotions` | > 60%                                                            | Verificar consistencia entre `was_on_promo` y la ventana promocional, revisar asignaciones CONTROL/TREATMENT y fechas             |
| **8. OTIF analítico de proveedor (proxy)**            | Indicador proxy de desempeño de proveedor basado en disponibilidad sostenida de productos ligados a ese proveedor | `items/proveedor sin gaps relevantes / total items/proveedor monitoreados`                                                                       | Semanal                   | `products` + `vendors` + base de gaps                     | > 95% en proveedores estratégicos                                | Revisar que la relación producto-proveedor exista y que los gaps no sean de demanda intermitente                                  |
| **9. Índice Compuesto de Productividad de Tienda**    | KPI compuesto que resume productividad comercial, experiencia y ejecución operativa en un solo score              | `0.4*(GMV/m² normalizado) + 0.2*(Ticket promedio normalizado) + 0.2*(Transacciones/m² normalizado) + 0.2*(Disponibilidad operativa normalizada)` | Semanal y mensual         | Derivado de KPIs 1, 2, 3 y 5                              | Score > 0.70; top quartile como benchmark                        | Revisar que todos los KPIs fuente estén actualizados y normalizados en la misma ventana temporal                                  |
| **10. Crecimiento Comp Sales de tiendas comparables** | Variación del GMV entre períodos comparables para tiendas operando en ambos períodos                              | `(GMV período actual - GMV período anterior) / GMV período anterior`                                                                             | Mensual y trimestral      | `transactions` + `stores` + `dim_date` lógica             | Positivo y superior al benchmark por formato                     | Verificar que las tiendas sean comparables, excluir tiendas nuevas y revisar semanas parciales                                    |
